# DisasterSeverity – BanglaBERT Large v3
**Targeting 0.85+ weighted F1** (v2 baseline: 0.78)

| # | Improvement | Expected gain |
|---|-------------|---------------|
| 1 | BanglaBERT Large (24-layer) replaces base (12-layer) | +3–4% |
| 2 | 10-fold CV instead of 5-fold | +0.5% |
| 3 | Layer-wise LR Decay (LLRD) | +0.5–1% |
| 4 | AWP (Adversarial Weight Perturbation) replaces FGM | +0.5–1% |
| 5 | EMA (Exponential Moving Average) weights | +0.5% |
| 6 | Data-driven logit adjustments (EDA-derived hard rules) | +0.5% |
| 7 | Optional: XLM-R Large ensemble (set USE_SECOND_MODEL=True) | +1–2% |
| – | fp16 mixed precision (speed only, no quality change) | – |

**Key EDA findings driving the rules:**
- Non Disaster → 100% Minimal in training data (hard rule retained)
- Wildfire + death keywords → P(Catastrophic):P(Severe) = 13:1 (logit boost)
- Flood + death keywords → P(Catastrophic):P(Severe) = 4.5:1 (logit boost)
- Earthquake + death keywords → P(Severe) dominates 2.3:1 over Catastrophic
- Catastrophic class is only 6.6% of training data — class weights critical
- Text length is a severity signal: Minimal=11 words avg, Catastrophic=24 words avg


In [ ]:
# ── Cell 1: Imports & Seed ─────────────────────────────────────────────────
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.optim import AdamW
from scipy.special import softmax as scipy_softmax
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import zipfile

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

seed_everything(42)

In [ ]:
# ── Cell 2: Configuration ──────────────────────────────────────────────────
class CFG:
    # ── Model
    # PRIMARY: BanglaBERT Large — biggest single improvement (+3-4%)
    # For multi-model ensemble (optional): train a second pass with xlm-roberta-large
    model_name   = "csebuetnlp/banglabert_large"
    model_name_2 = "xlm-roberta-large"          # set USE_SECOND_MODEL=True to activate
    base_path    = "/kaggle/input/competitions/datathon-iiuc-cse-fest-2026/DisasterSeverity/"

    USE_SECOND_MODEL = False   # flip to True if you have GPU budget for +1-2%

    # ── Tokenisation
    # EDA: median 12 words, p99=51 words, only 0.6% texts exceed 128 subword tokens.
    # max_len=128 is sufficient; increasing to 200 wastes memory with no gain.
    max_len = 128

    # ── Training (tuned for Large models on T4/P100)
    epochs       = 6
    batch_size   = 8
    grad_accum   = 2           # effective batch = 16
    lr           = 1e-5        # lower than base (large models need gentler LR)
    warmup_ratio = 0.10
    weight_decay = 0.01
    llrd_decay   = 0.9         # layer-wise LR decay factor

    # ── Loss
    # Reduced from 0.1 — large models already well-regularised
    label_smoothing = 0.05

    # ── Regularisation
    use_rdrop   = True
    rdrop_alpha = 0.30         # reduced from 0.5; large model needs less KL push
    use_awp     = True         # AWP replaces FGM: perturbs weights, not just embeddings
    awp_lr      = 0.1
    awp_eps     = 1e-4
    awp_start_epoch = 2        # wait 2 epochs before adversarial training

    # ── EMA (Exponential Moving Average of weights — free +0.5%)
    use_ema   = True
    ema_decay = 0.999

    # ── Cross-Validation
    n_folds = 10               # 10-fold: each model sees 90% of data vs 80%
    seed    = 42

    # ── Pseudo-Labelling
    use_pseudo       = True
    pseudo_threshold = 0.92
    pseudo_epochs    = 2
    pseudo_lr        = 5e-6

label_map         = {"Minimal": 0, "Mild": 1, "Moderate": 2, "Severe": 3, "Catastrophic": 4}
reverse_label_map = {v: k for k, v in label_map.items()}

In [ ]:
# ── Cell 3: Data Loading ───────────────────────────────────────────────────
# EDA summary (training data, 2800 samples):
#   Non Disaster  → 100% Minimal (hard rule)
#   Tropical Storm → 55% Mild (model learns this)
#   Human Damage  → 55% Moderate
#   Earthquake    → 47% Moderate
#   Flood         → 47% Moderate
#   Wildfire      → 40% Severe
#   Landslides    → 36% Severe
#   Drought       → 33% Severe
#   Catastrophic  → only 6.6% of training data — critically underrepresented

print("Loading data...")
train = pd.read_csv(f"{CFG.base_path}train.csv")
test  = pd.read_csv(f"{CFG.base_path}test.csv")
val   = pd.read_csv(f"{CFG.base_path}validation.csv")
train = pd.concat([train, val]).reset_index(drop=True)

# Text template: proven effective in v2 (category prefix guides the model)
train["text"] = train["category"] + ": " + train["context"].fillna("")
test["text"]  = test["category"]  + ": " + test["context"].fillna("")

train["label_id"] = train["label"].map(label_map)

skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(train, train["label_id"])):
    train.loc[val_idx, "fold"] = fold

tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=CFG.max_len,
    )

test_dataset   = Dataset.from_pandas(test[["text"]])
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

# Class weights — Catastrophic (6.6%) gets ~3× boost; keeps recall high
class_counts  = np.bincount(train["label_id"].values, minlength=5).astype(float)
CLASS_WEIGHTS = (len(train) / (5 * class_counts)).tolist()
print("Class weights:", {k: round(CLASS_WEIGHTS[v], 3) for k, v in label_map.items()})
print(f"Train: {len(train)} | Test: {len(test)}")

In [ ]:
# ── Cell 4: AWP + EMA + LLRD utilities ────────────────────────────────────

class AWP:
    """
    Adversarial Weight Perturbation (stronger than FGM).
    Perturbs MODEL WEIGHTS in gradient direction → broader robustness.
    Paper: AWP (Wu et al. 2020), typical gain +0.5-1% on NLP tasks.
    """
    def __init__(self, model, adv_lr=0.1, adv_eps=1e-4):
        self.model   = model
        self.adv_lr  = adv_lr
        self.adv_eps = adv_eps
        self.backup  = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm1 = torch.norm(param.grad)
                norm2 = torch.norm(param.data)
                if norm1 != 0 and not torch.isnan(norm1):
                    r_at = self.adv_lr * param.grad / norm1 * norm2
                    param.data.add_(r_at)
                    param.data = self._project(name, param.data)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

    def _project(self, param_name, param_data):
        r = param_data - self.backup[param_name]
        if torch.norm(r) > self.adv_eps:
            r = self.adv_eps * r / torch.norm(r)
        return self.backup[param_name] + r


class EMA:
    """
    Exponential Moving Average of model weights.
    Equivalent to ensembling all checkpoints across training. +0.5% typical.
    Applied during eval/predict; model trains on live weights.
    """
    def __init__(self, model, decay=0.999):
        self.model  = model
        self.decay  = decay
        self.shadow = {}
        self.backup = {}
        self._register()

    def _register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (
                    (1 - self.decay) * param.data + self.decay * self.shadow[name]
                )

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}


def get_llrd_params(model, lr, weight_decay=0.01, llrd=0.9):
    """
    Layer-wise LR Decay.
    Bottom layers (general language) → tiny LR (preserves pre-training).
    Top layers + head → full LR.
    
    Gain: ~0.5-1% F1 by preventing catastrophic forgetting.
    """
    no_decay = ["bias", "LayerNorm.weight"]

    # Works for both BanglaBERT (bert) and XLM-R (roberta) families
    if hasattr(model, 'bert'):
        base = model.bert
    elif hasattr(model, 'roberta'):
        base = model.roberta
    else:
        base = model.base_model

    encoder_layers = list(base.encoder.layer)
    n = len(encoder_layers)
    params = []

    # Embeddings → floor LR
    emb_lr = lr * (llrd ** n)
    params += [
        {"params": [p for nm, p in base.embeddings.named_parameters()
                    if not any(nd in nm for nd in no_decay)],
         "lr": emb_lr, "weight_decay": weight_decay},
        {"params": [p for nm, p in base.embeddings.named_parameters()
                    if any(nd in nm for nd in no_decay)],
         "lr": emb_lr, "weight_decay": 0.0},
    ]

    # Transformer layers → exponentially increasing LR
    for i, layer in enumerate(encoder_layers):
        layer_lr = lr * (llrd ** (n - i - 1))
        params += [
            {"params": [p for nm, p in layer.named_parameters()
                        if not any(nd in nm for nd in no_decay)],
             "lr": layer_lr, "weight_decay": weight_decay},
            {"params": [p for nm, p in layer.named_parameters()
                        if any(nd in nm for nd in no_decay)],
             "lr": layer_lr, "weight_decay": 0.0},
        ]

    # Pooler if exists
    if hasattr(base, 'pooler') and base.pooler is not None:
        params += [
            {"params": [p for nm, p in base.pooler.named_parameters()
                        if not any(nd in nm for nd in no_decay)],
             "lr": lr, "weight_decay": weight_decay},
            {"params": [p for nm, p in base.pooler.named_parameters()
                        if any(nd in nm for nd in no_decay)],
             "lr": lr, "weight_decay": 0.0},
        ]

    # Classifier head → full LR
    params += [
        {"params": [p for nm, p in model.classifier.named_parameters()
                    if not any(nd in nm for nd in no_decay)],
         "lr": lr, "weight_decay": weight_decay},
        {"params": [p for nm, p in model.classifier.named_parameters()
                    if any(nd in nm for nd in no_decay)],
         "lr": lr, "weight_decay": 0.0},
    ]
    return params


def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    _, _, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [ ]:
# ── Cell 5: AdvancedTrainer V3 ─────────────────────────────────────────────

class AdvancedTrainerV3(Trainer):
    """
    V3 stacks over V2:
      R-Drop (kept)
      AWP replacing FGM  (+0.5%)
      EMA weights        (+0.5%)
      LLRD optimizer     (+0.7%)
      AWP delayed start  (avoids attacking an unstable model early)
    """

    def __init__(self, *args, epoch_steps=0, **kwargs):
        super().__init__(*args, **kwargs)
        self.epoch_steps = epoch_steps   # steps per epoch, for AWP timing
        self._step       = 0
        self._ema        = None

    def _init_ema(self):
        if CFG.use_ema and self._ema is None:
            self._ema = EMA(self.model, decay=CFG.ema_decay)

    def _ce(self, logits, labels):
        wt = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(logits.device)
        return nn.CrossEntropyLoss(weight=wt, label_smoothing=CFG.label_smoothing)(
            logits, labels
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        if model.training and CFG.use_rdrop and not getattr(self, "_awp_mode", False):
            out1 = model(**inputs)
            out2 = model(**inputs)
            ce   = (self._ce(out1.logits, labels) + self._ce(out2.logits, labels)) / 2
            p1   = F.softmax(out1.logits, dim=-1)
            p2   = F.softmax(out2.logits, dim=-1)
            kl   = (
                F.kl_div(out1.logits.log_softmax(-1), p2, reduction="batchmean") +
                F.kl_div(out2.logits.log_softmax(-1), p1, reduction="batchmean")
            ) / 2
            loss = ce + CFG.rdrop_alpha * kl
            return (loss, out1) if return_outputs else loss
        else:
            out  = model(**inputs)
            loss = self._ce(out.logits, labels)
            return (loss, out) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None, **kwargs):
        self._init_ema()
        self._step += 1
        model.train()
        inputs = self._prepare_inputs(inputs)
        labels = inputs["labels"].clone()

        # Normal forward + backward
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)
        if self.args.gradient_accumulation_steps > 1:
            loss = loss / self.args.gradient_accumulation_steps
        self.accelerator.backward(loss)

        # AWP: delayed start (attack only after model is stable)
        awp_start = self.epoch_steps * CFG.awp_start_epoch
        if CFG.use_awp and self._step >= awp_start:
            awp = AWP(model, adv_lr=CFG.awp_lr, adv_eps=CFG.awp_eps)
            awp.attack()
            inputs["labels"] = labels
            self._awp_mode = True
            with self.compute_loss_context_manager():
                loss_adv = self.compute_loss(model, inputs)
            self._awp_mode = False
            if self.args.gradient_accumulation_steps > 1:
                loss_adv = loss_adv / self.args.gradient_accumulation_steps
            self.accelerator.backward(loss_adv)
            awp.restore()

        # EMA update after each gradient step
        if CFG.use_ema and self._ema is not None:
            self._ema.update()

        return loss.detach()

    def create_optimizer(self):
        """Inject LLRD into the optimizer."""
        if self.optimizer is None:
            params = get_llrd_params(
                self.model,
                lr=self.args.learning_rate,
                weight_decay=self.args.weight_decay,
                llrd=CFG.llrd_decay,
            )
            self.optimizer = AdamW(params)
        return self.optimizer

    def evaluate(self, *args, **kwargs):
        """Eval on EMA weights → better validation scores for checkpoint selection."""
        if CFG.use_ema and self._ema is not None:
            self._ema.apply_shadow()
        result = super().evaluate(*args, **kwargs)
        if CFG.use_ema and self._ema is not None:
            self._ema.restore()
        return result

    def predict(self, *args, **kwargs):
        """Predict on EMA weights."""
        if CFG.use_ema and self._ema is not None:
            self._ema.apply_shadow()
        result = super().predict(*args, **kwargs)
        if CFG.use_ema and self._ema is not None:
            self._ema.restore()
        return result

In [ ]:
# ── Cell 6: 10-Fold Training ───────────────────────────────────────────────
def run_cv(model_name, tokenizer, tokenized_test, tag=""):
    """Run 10-fold CV for a given model. Returns (test_logits, oof_logits)."""
    test_preds = []
    oof_preds  = np.zeros((len(train), 5))

    _tokenize = lambda examples: tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=CFG.max_len,
    )

    for fold in range(CFG.n_folds):
        print(f"\n{'='*18} {tag}FOLD {fold+1}/{CFG.n_folds} {'='*18}")

        trn_df = train[train["fold"] != fold].reset_index(drop=True)
        val_df = train[train["fold"] == fold].reset_index(drop=True)

        tok_trn = (
            Dataset.from_pandas(trn_df[["text", "label_id"]]
                                .rename(columns={"label_id": "label"}))
            .map(_tokenize, batched=True)
        )
        tok_val = (
            Dataset.from_pandas(val_df[["text", "label_id"]]
                                .rename(columns={"label_id": "label"}))
            .map(_tokenize, batched=True)
        )

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=5
        )

        epoch_steps = len(trn_df) // (CFG.batch_size * CFG.grad_accum)

        args = TrainingArguments(
            output_dir                  = f"/kaggle/working/{tag}fold_{fold}",
            eval_strategy               = "epoch",
            save_strategy               = "epoch",
            learning_rate               = CFG.lr,
            per_device_train_batch_size = CFG.batch_size,
            per_device_eval_batch_size  = CFG.batch_size * 2,
            gradient_accumulation_steps = CFG.grad_accum,
            num_train_epochs            = CFG.epochs,
            warmup_ratio                = CFG.warmup_ratio,
            lr_scheduler_type           = "cosine",
            weight_decay                = CFG.weight_decay,
            load_best_model_at_end      = True,
            metric_for_best_model       = "f1",
            greater_is_better           = True,
            report_to                   = "none",
            save_total_limit            = 1,
            fp16                        = True,
        )

        trainer = AdvancedTrainerV3(
            model         = model,
            args          = args,
            train_dataset = tok_trn,
            eval_dataset  = tok_val,
            compute_metrics = compute_metrics,
            epoch_steps   = epoch_steps,
        )

        trainer.train()

        # OOF
        val_logits = trainer.predict(tok_val).predictions
        oof_preds[val_df.index] = scipy_softmax(val_logits, axis=-1)

        print(f"  → Predicting test...")
        test_logits = trainer.predict(tokenized_test).predictions
        test_preds.append(test_logits)

    return np.mean(test_preds, axis=0), oof_preds


# ── Run Model 1 (BanglaBERT Large) ────────────────────────────────────────
tokenizer1     = AutoTokenizer.from_pretrained(CFG.model_name)
test_ds1       = Dataset.from_pandas(test[["text"]]).map(
    lambda ex: tokenizer1(ex["text"], padding="max_length",
                          truncation=True, max_length=CFG.max_len), batched=True)
logits1, oof1  = run_cv(CFG.model_name, tokenizer1, test_ds1, tag="bb_")

# ── Run Model 2 (XLM-R Large) — optional ──────────────────────────────────
if CFG.USE_SECOND_MODEL:
    print("\n\n" + "="*60)
    print("  SECOND MODEL: xlm-roberta-large")
    print("="*60)
    tokenizer2 = AutoTokenizer.from_pretrained(CFG.model_name_2)
    test_ds2   = Dataset.from_pandas(test[["text"]]).map(
        lambda ex: tokenizer2(ex["text"], padding="max_length",
                              truncation=True, max_length=CFG.max_len), batched=True)
    logits2, oof2 = run_cv(CFG.model_name_2, tokenizer2, test_ds2, tag="xlm_")
    # Equal-weight ensemble (both models are strong; can weight by OOF F1 if desired)
    final_logits = 0.55 * logits1 + 0.45 * logits2
    print("\nUsing ensemble: 55% BanglaBERT-Large + 45% XLM-R-Large")
else:
    final_logits = logits1
    print("\nUsing BanglaBERT-Large only (set USE_SECOND_MODEL=True to ensemble)")

In [ ]:
# ── Cell 7: Data-Driven Post-Processing ───────────────────────────────────
#
# Rules derived from training data EDA (death keyword × category):
#
#   Wildfire + death keywords → P(Cat) : P(Sev) = 13 : 1  → boost Cat by ln(13)≈2.6
#   Flood    + death keywords → P(Cat) : P(Sev) = 4.5 : 1 → boost Cat by ln(4.5)≈1.5
#   Earthquake+death keywords → P(Sev) : P(Cat) = 2.3 : 1 → boost Sev by ln(2.3)≈0.8
#   Landslides+death keywords → P(Sev) : P(Cat) = 3.8 : 1 → boost Sev by ln(3.8)≈1.3
#
# Applied as log-odds adjustments to the ensemble logits (soft rules, not hard overrides).
# Non Disaster rule remains a hard override (100% verified in training data).

DEATH_KWS = ['মৃত', 'নিহত', 'নিখোঁজ', 'হতাহত', 'মৃতদেহ']

def has_death_kw(text):
    return any(w in str(text) for w in DEATH_KWS)

adjusted_logits = final_logits.copy()

for idx, row in test.iterrows():
    cat     = row["category"]
    context = row["context"]
    if not has_death_kw(context):
        continue

    if cat == "Wildfire":
        adjusted_logits[idx, label_map["Catastrophic"]] += 2.6
    elif cat == "Flood":
        adjusted_logits[idx, label_map["Catastrophic"]] += 1.5
    elif cat == "Earthquake":
        adjusted_logits[idx, label_map["Severe"]]       += 0.8
    elif cat == "Landslides":
        adjusted_logits[idx, label_map["Severe"]]       += 1.3

final_preds       = np.argmax(adjusted_logits, axis=-1)
test["label"]     = [reverse_label_map[p] for p in final_preds]

# ── Hard Rule: Non Disaster → Minimal (100% verified, overrides everything)
test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"
print("Hard rule applied: Non Disaster → Minimal")
print("\nPrediction distribution (post-processing):")
print(test["label"].value_counts())

In [ ]:
# ── Cell 8: Pseudo-Labelling ───────────────────────────────────────────────
if CFG.use_pseudo:
    print("\n── Pseudo-Labelling ──")
    probs     = scipy_softmax(adjusted_logits, axis=-1)
    max_probs = probs.max(axis=-1)

    # Exclude Non Disaster (rule-based, pseudo would just add noise)
    nd_mask   = (test["category"] == "Non Disaster").values
    confident = (max_probs >= CFG.pseudo_threshold) & (~nd_mask)
    print(f"High-confidence samples: {confident.sum()}/{(~nd_mask).sum()} "
          f"(threshold={CFG.pseudo_threshold})")

    if confident.sum() > 50:
        pseudo_df            = test[confident].copy()
        pseudo_df["label_id"] = final_preds[confident]

        full_train = pd.concat(
            [train[["text", "label_id"]], pseudo_df[["text", "label_id"]]],
            ignore_index=True,
        )
        print(f"Expanded train: {len(train)} → {len(full_train)} samples")

        tok_full = (
            Dataset.from_pandas(full_train.rename(columns={"label_id": "label"}))
            .map(lambda ex: tokenizer1(ex["text"], padding="max_length",
                                       truncation=True, max_length=CFG.max_len),
                 batched=True)
        )

        pseudo_model = AutoModelForSequenceClassification.from_pretrained(
            CFG.model_name, num_labels=5
        )
        pseudo_args = TrainingArguments(
            output_dir                  = "/kaggle/working/pseudo",
            num_train_epochs            = CFG.pseudo_epochs,
            per_device_train_batch_size = CFG.batch_size,
            per_device_eval_batch_size  = CFG.batch_size * 2,
            gradient_accumulation_steps = CFG.grad_accum,
            learning_rate               = CFG.pseudo_lr,
            warmup_ratio                = CFG.warmup_ratio,
            lr_scheduler_type           = "cosine",
            weight_decay                = CFG.weight_decay,
            save_strategy               = "no",
            report_to                   = "none",
            fp16                        = True,
        )
        pseudo_trainer = AdvancedTrainerV3(
            model         = pseudo_model,
            args          = pseudo_args,
            train_dataset = tok_full,
            compute_metrics = compute_metrics,
        )
        pseudo_trainer.train()
        pseudo_logits = pseudo_trainer.predict(test_ds1).predictions

        # Blend: 75% CV ensemble + 25% pseudo
        blended      = 0.75 * adjusted_logits + 0.25 * pseudo_logits
        final_preds  = np.argmax(blended, axis=-1)
        test["label"] = [reverse_label_map[p] for p in final_preds]
        test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"
        print("Blended (75% CV + 25% pseudo). Non Disaster rule re-applied.")
    else:
        print("Too few confident samples; skipping.")

In [ ]:
# ── Cell 9: Save Submission ────────────────────────────────────────────────
submission = test[["image_id", "label"]]
submission.to_csv("submission.csv", index=False)

with zipfile.ZipFile("submission.zip", "w") as z:
    z.write("submission.csv", arcname="submission.csv")

print("\n✅ submission.zip ready for upload.")
print("\nFinal prediction distribution:")
print(submission["label"].value_counts())
print()
print(submission.head(10))